In [ ]:
import json
from typing import *
from loader import load_training_problem, list_training_problems

data = list_training_problems()
problem_id = data[0]
example = load_training_problem(problem_id)

In [ ]:
PROMPT_V2 = (
    "Solve task {task_id}\n\n"
    "INPUT:\n{input}\n"
    "OUTPUT PLACEHOLDER:\n{placeholder}\n"
    "OUTPUT:"
)

In [ ]:
import numpy as np

grid_placeholder = np.zeros((3,3)).astype(int)
placeholder_rows = "\n".join([" ".join(str(c) for c in row) for row in grid_placeholder])

In [ ]:
def grid_to_row_strings(grid: List[List[int]]) -> List[str]:
    """
    Convert a grid (list of lists) to row-string format.
    
    Args:
        grid: Grid as list of lists of integers
        
    Returns:
        List of strings, where each string represents a row
    """
    return [' '.join(map(str, row)) for row in grid]

def _format_single_prompt(input_grid: List[List[int]], placeholder_rows: str, task_id: str) -> str:
    """Format a single-input prompt with PROMPT_V2."""
    input_str = "\n".join(grid_to_row_strings(input_grid))
    return PROMPT_V2.format(task_id=task_id, input=input_str, placeholder=placeholder_rows)

In [ ]:
prompt_list = []
output_list = []
for task_id in data:
    problem = load_training_problem(problem_id)
    for sample in problem["train"]:
        formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
        formatted_output = grid_to_row_strings(sample["output"])
        prompt_list.append(formatted_prompt)
        output_list.append(formatted_output)
    for sample in problem["test"]:
        formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
        formatted_output = grid_to_row_strings(sample["output"])
        prompt_list.append(formatted_prompt)
        output_list.append(formatted_output)        

In [ ]:
import tiktoken

# Initialize tokenizer (using cl100k_base which is used by GPT-4)
tokenizer = tiktoken.get_encoding("cl100k_base")

# Count tokens for prompts
prompt_token_counts = [len(tokenizer.encode(prompt)) for prompt in prompt_list]

# Count tokens for outputs (join list of strings first)
output_token_counts = [len(tokenizer.encode("\n".join(output))) for output in output_list]

# Calculate statistics for prompts
prompt_stats = {
    "mean": np.mean(prompt_token_counts),
    "median": np.median(prompt_token_counts),
    "max": np.max(prompt_token_counts),
    "min": np.min(prompt_token_counts)
}

# Calculate statistics for outputs
output_stats = {
    "mean": np.mean(output_token_counts),
    "median": np.median(output_token_counts),
    "max": np.max(output_token_counts),
    "min": np.min(output_token_counts)
}

print("Prompt token statistics:")
for stat, value in prompt_stats.items():
    print(f"  {stat}: {value:.2f}")

print("\nOutput token statistics:")
for stat, value in output_stats.items():
    print(f"  {stat}: {value:.2f}")


In [ ]:
from vllm import LLM, SamplingParams
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)
llm = LLM(model="Qwen/Qwen3-4B-Instruct-2507-FP8")

In [ ]:
outputs = llm.generate([prompt_list[0]], sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
train_problems = {"conversations":[]}
test_problems = {"conversations":[]}
problem = load_training_problem(data[0])
for sample in problem["train"]:
    formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
    formatted_output = "\n".join(grid_to_row_strings(sample["output"]))
    train_problem = []
    user_content = {"role":"user", "content":""}
    user_content["content"] = formatted_prompt
    assistant_content = {"role":"assistant", "content":""}
    assistant_content["content"] = formatted_output
    train_problem.append(user_content)
    train_problem.append(assistant_content)
    train_problems["conversations"].append(train_problem)
for sample in problem["test"]:
    formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
    formatted_output = "\n".join(grid_to_row_strings(sample["output"]))
    test_problem = []
    user_content = {"role":"user", "content":""}
    user_content["content"] = formatted_prompt
    assistant_content = {"role":"assistant", "content":""}
    assistant_content["content"] = formatted_output
    test_problem.append(user_content)
    test_problem.append(assistant_content)
    test_problems["conversations"].append(test_problem)

In [ ]:
import json
with open('data.json', 'w') as f:
    json.dump(train_problems, f)

In [ ]:
import os
import platform
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

def pick_attn_impl() -> str:
    if platform.system() == "Linux":
        try:
            import importlib
            importlib.import_module("flash_attn")
            return "flash_attention_2"
        except Exception:
            return "sdpa"
    return "sdpa"

def run_sft(
    dataset_path: str,
    output_dir: str = "qwen3_4b_singled_out_sft",
    base_model: str = "Qwen/Qwen2.5-0.5B-Instruct",
    learning_rate: float = 8e-5,
    num_train_epochs: int = 20,
    use_compile: bool = False,
):
    """Run minimal SFT on the singled-out dataset with LoRA."""

    def formatting_prompts_func(examples):
        texts = tokenizer.apply_chat_template(examples, tokenize = False, add_generation_prompt = False)
        return { "text" : texts, }
    load_dotenv()
    if os.getenv("HF_TOKEN"):
        try:
            login(os.getenv("HF_TOKEN"))
        except Exception:
            pass
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    attn_impl = pick_attn_impl()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-0.5B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = compute_dtype,
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    )
    # Dataset
    with open("data.json") as f:
        raw = json.load(f)
    data = tokenizer.apply_chat_template(
        raw["conversations"],
        tokenize = False,
    )
    import pandas as pd
    data = pd.Series(data)
    data.name = "text"
    
    from datasets import Dataset
    dataset = Dataset.from_pandas(pd.DataFrame(data))
    dataset = dataset.shuffle(seed = 3407)

    # Trainer
    args = SFTConfig(
        #loss_type="dft",
        output_dir=output_dir,
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=25,
        save_steps=200,
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
        optim="paged_adamw_8bit",
        ddp_find_unused_parameters=False,
        max_grad_norm=None,
    )

    from trl import SFTTrainer
    from transformers import DataCollatorForSeq2Seq
    
    trainer = SFTTrainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=8192,
    )
        
    print("[sft] Starting training...")
    trainer.train()
    print("[sft] Saving final adapter...")
    trainer.save_model(os.path.join(output_dir, "final"))
    try:
        tokenizer.save_pretrained(os.path.join(output_dir, "final"))
    except Exception:
        pass
    
    # Simple inference evaluation after SFT
    print("[sft] Running simple inference evaluation...")
    try:
        sample_data = raw["conversations"][0][0]["content"]
        messages = [
            {"role": "user", "content": sample_data},
        ]
        from unsloth.chat_templates import get_chat_template
        FastLanguageModel.for_inference(model) # Enable native 2x faster inference
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
        
        outputs = model.generate(input_ids = inputs, max_new_tokens = 2048, use_cache = True)
        print(tokenizer.batch_decode(outputs))
    except Exception as e:
        print(f"[sft] Evaluation failed: {e}")

    
    return os.path.join(output_dir, "final")

In [ ]:
sft_path = run_sft("data.json")

In [ ]:
import re
from typing import List, Optional


def check_array(output_string: str) -> bool:
    if not output_string or not isinstance(output_string, str):
        return False
    response = output_string.strip()
    if not response:
        return False
    if '\n' in response:
        grid_match = re.search(r'[0-9\n\s]+', response)
        if not grid_match:
            return False
        grid_str = grid_match.group()
        try:
            rows = grid_str.split('\n')
            if not rows:
                return False
            grid = []
            expected_width = None
            for row in rows:
                if not row.strip():
                    return False
                parts = row.strip().split()
                if len(parts) > 1:
                    try:
                        grid_row = [int(p) for p in parts if p.strip()]
                    except ValueError:
                        return False
                else:
                    if not row.strip().isdigit():
                        return False
                    grid_row = [int(char) for char in row.strip()]
                if any(digit < 0 or digit > 9 for digit in grid_row):
                    return False
                if expected_width is None:
                    expected_width = len(grid_row)
                elif len(grid_row) != expected_width:
                    return False
                grid.append(grid_row)
            return len(grid) > 0 and len(grid[0]) > 0
        except (ValueError, IndexError):
            return False
    return False


def check_value(output_string: str, expected_value: List[List[int]]) -> bool:
    if not isinstance(expected_value, list) or not expected_value:
        return False
    if not check_array(output_string):
        return False
    parsed_grid = parse_grid_from_string(output_string)
    if parsed_grid is None:
        return False
    return parsed_grid == expected_value


def parse_grid_from_string(output_string: str) -> Optional[List[List[int]]]:
    if not output_string or not isinstance(output_string, str):
        return None
    response = output_string.strip()
    if not response:
        return None
    if '\n' in response:
        grid_match = re.search(r'[0-9\n\s]+', response)
        if not grid_match:
            return None
        grid_str = grid_match.group()
        try:
            rows = grid_str.split('\n')
            grid = []
            for row in rows:
                if not row.strip():
                    continue
                parts = row.strip().split()
                if len(parts) > 1:
                    try:
                        grid_row = [int(p) for p in parts if p.strip()]
                    except ValueError:
                        return None
                else:
                    if not row.strip().isdigit():
                        return None
                    grid_row = [int(char) for char in row.strip()]
                if any(digit < 0 or digit > 9 for digit in grid_row):
                    return None
                grid.append(grid_row)
            return grid if grid else None
        except (ValueError, IndexError):
            return None
    return None

def reward_function(
    completions: List[str], 
    expected_output: List[str], 
    **kwargs: Any
) -> List[float]:
    rewards = []
    for completion, expected in zip(completions, expected_output, strict=False):
        if not check_array(completion):
            rewards.append(-1.0)
            continue
        if check_value(completion, parse_grid_from_string(expected)):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_sft/final", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = torch.bfloat16,
        load_in_4bit = True,
    )

model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
)

with open("data.json") as f:
        raw = json.load(f)
sample_data = raw["conversations"][0][0]["content"]
messages = [
            {"role": "user", "content": sample_data},
]
from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 2048, use_cache = True)
generated_tokens = outputs[:, inputs.shape[-1]:]
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(decoded[0])
print(check_array(decoded[0]))
print(check_value(decoded[0], parse_grid_from_string(raw["conversations"][0][1]["content"])))
print(check_value(raw["conversations"][0][1]["content"], parse_grid_from_string(raw["conversations"][0][1]["content"])))
print(reward_function(decoded, [raw["conversations"][0][1]["content"]]))

In [ ]:
def convert_conversations(raw_json):
    result = []
    for convo in raw_json["conversations"]:
        # Expecting [ {"role":"user"}, {"role":"assistant"} ]
        user_msg = convo[0]["content"]
        assistant_msg = convo[1]["content"]
        result.append({
            "prompt": [
                {"role": "user", "content": user_msg}
            ],
            "expected_output": assistant_msg
        })
    return result

def run_rl(
    #base_model: str,
    #lora_path: str,
    #dataset_path: str,
    output_dir: str = "qwen3_4b_singled_out_rl",
    learning_rate: float = 1e-5,
    num_train_epochs: int = 1,
    grad_accum: int = 4,
    num_generations: int = 4,
):
    """Run minimal GRPO on top of SFT LoRA using the same dataset."""
    import platform
    import torch
    from datasets import load_dataset, Dataset
    from peft import PeftModel
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from trl import GRPOConfig, GRPOTrainer
    from unsloth import FastLanguageModel
    import torch
    max_seq_length = 2048 # Can increase for longer reasoning traces
    lora_rank = 32 # Larger rank = smarter, but slower
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_sft/final",
        max_seq_length = max_seq_length,
        load_in_4bit = False, # False for LoRA 16bit
        fast_inference = True, # Enable vLLM fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.2, # Reduce if out of memory
    )
    with open("data.json") as f:
        raw = json.load(f)
    converted = convert_conversations(raw)
    dataset = Dataset.from_list(converted)  
    print(dataset)
    from vllm import SamplingParams
    vllm_sampling_params = SamplingParams(
        min_p = 0.1,
        top_p = 1.0,
        top_k = -1,
        seed = 3407,
        stop = [tokenizer.eos_token],
        include_stop_str_in_output = True,
    )
    
    from trl import GRPOConfig, GRPOTrainer
    training_args = GRPOConfig(
        vllm_sampling_params = vllm_sampling_params,
        importance_sampling_level="sequence",
        loss_type="grpo",
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=grad_accum,
        beta=0.04,
        epsilon=3e-4,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=200,
        optim="paged_adamw_8bit",
        report_to="none",
        num_generations=num_generations,
        max_prompt_length=4096,
        max_completion_length=2048,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            reward_function
        ],
        args = training_args,
        train_dataset = dataset,
    
        # For optional training + evaluation
        # train_dataset = new_dataset["train"],
        # eval_dataset = new_dataset["test"],
    )
    trainer.train()
    trainer.save_model(os.path.join(output_dir, "final"))
    try:
        tokenizer.save_pretrained(os.path.join(output_dir, "final"))
    except Exception:
        pass
    return os.path.join(output_dir, "final")

In [ ]:
run_rl()